In [58]:
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from entsoe import EntsoePandasClient
from entsoe.mappings import Area, lookup_area, NEIGHBOURS
from entsoe.exceptions import NoMatchingDataError

# Setup paths
cwd = Path.cwd()
api_file = cwd.parent.parent / "entsoe-api.txt"
if not api_file.exists():
    api_file = Path("entsoe-api.txt")

# Initialize global flags
API_AVAILABLE = False
API_TOKEN = ""

if api_file.exists():
    try:
        API_TOKEN = api_file.read_text(encoding="utf-8").strip()
        if API_TOKEN and "<YOUR" not in API_TOKEN:
            client = EntsoePandasClient(api_key=API_TOKEN)
            API_AVAILABLE = True
    except Exception:
        pass

# ==========================================================
# DATA LOADING CONTROLLER (ONLINE VS DEMO OFFLINE MODE)
# ==========================================================
data_results = {}
local_tz = "Europe/Berlin" 

if API_AVAILABLE:
    print("🌐 API Key detected! Running in ONLINE MODE.")
    
    # [STEP 1] Select Origin Zone
    all_zones = sorted([area.name for area in Area])
    print("\nAvailable Origin Bidding Zones:\n" + ", ".join(all_zones))
    country_code = input("\nEnter origin bidding zone (e.g., DK_1, DE): ").strip().upper()
    if not hasattr(Area, country_code):
        raise ValueError(f"Invalid bidding zone: '{country_code}'")
        
    country_code_from = country_code
    area_origin = lookup_area(country_code)
    local_tz = area_origin.tz
    
    # [STEP 2] Select Destination Zone
    possible_neighbors = sorted(NEIGHBOURS.get(country_code, []))
    print(f"\nPossible Neighbors for {country_code}:\n" + ", ".join(possible_neighbors))
    country_code_to = input("\nEnter destination bidding zone: ").strip().upper()
    
    # [STEP 3] Date Range Configuration
    print("\nEnter dates using pattern YYYYMMDD (e.g., 20260920)")
    start_input = input("START date: ").strip()
    end_input = input("END date: ").strip()
    
    start = pd.Timestamp(start_input, tz=local_tz)
    end = pd.Timestamp(end_input, tz=local_tz)
    
    def safe_query(func_name, func, *args, **kwargs):
        try:
            res = func(*args, **kwargs)
            data_results[func_name] = res
        except NoMatchingDataError:
            print(f"❌ No matching data found for: {func_name}")
        except Exception as e:
            print(f"⚠️ Error fetching {func_name}: {e}")

    print("\n⏳ Fetching live data from ENTSO-E...")
    safe_query('day_ahead_prices', client.query_day_ahead_prices, country_code, start=start, end=end)
    safe_query('generation', client.query_generation, country_code, start=start, end=end, psr_type=None)
    safe_query('generation_forecast', client.query_generation_forecast, country_code, start=start, end=end) # ออนไลน์คีย์พยากรณ์
    safe_query('load', client.query_load, country_code, start=start, end=end)
    safe_query('net_position', client.query_net_position, country_code, start=start, end=end)
    safe_query('crossborder_flows', client.query_crossborder_flows, country_code_from, country_code_to, start=start, end=end)

else:
    print("🛑 API Key not found. Switching to OFFLINE DEMO MODE.")
    print("📊 Loading pre-saved sample data for DK_1 ➔ DE_LU (2026-09-20 to 2026-09-25)...")
    
    country_code = "DK_1 (Demo)"
    country_code_to = "DE_LU"
    
    files = {
        'day_ahead_prices': 'demo_day_ahead_prices.csv',
        'generation': 'demo_generation.csv',
        'generation_forecast': 'demo_generation_forecast.csv', # เดโมคีย์พยากรณ์
        'load': 'demo_load.csv',
        'net_position': 'demo_net_position.csv',
        'crossborder_flows': 'demo_crossborder_flows.csv'
    }
    
    for key, filename in files.items():
        csv_path = Path(filename)
        if csv_path.exists():
            df = pd.read_csv(csv_path, index_col=0, parse_dates=True)
            if key in ['day_ahead_prices', 'generation_forecast', 'load', 'net_position', 'crossborder_flows'] and isinstance(df, pd.DataFrame):
                data_results[key] = df.iloc[:, 0] if df.shape[1] == 1 else df
            else:
                data_results[key] = df
        else:
            print(f"❌ Demo file missing: '{filename}'.")

🌐 API Key detected! Running in ONLINE MODE.

Available Origin Bidding Zones:
AL, AT, BA, BE, BG, BY, CH, CWE, CY, CZ, CZ_DE_SK, DE, DE_50HZ, DE_AMPRION, DE_AMP_LU, DE_AT_LU, DE_LU, DE_TENNET, DE_TRANSNET, DK, DK_1, DK_1_NO_1, DK_2, DK_CA, EE, ES, FI, FR, GB, GB_ELECLINK, GB_IFA, GB_IFA2, GE, GR, HR, HU, IE, IE_SEM, IS, IT, IT_BRNN, IT_CALA, IT_CNOR, IT_CSUD, IT_FOGN, IT_GR, IT_MACRO_NORTH, IT_MACRO_SOUTH, IT_MALTA, IT_NORD, IT_NORD_AT, IT_NORD_CH, IT_NORD_FR, IT_NORD_SI, IT_PRGP, IT_ROSN, IT_SACO_AC, IT_SACO_DC, IT_SARD, IT_SICI, IT_SUD, LT, LU, LV, MD, ME, MK, MT, NIE, NL, NO, NO_1, NO_1A, NO_2, NO_2A, NO_2_NSL, NO_3, NO_4, NO_5, PL, PL_CZ, PT, RO, RS, RU, RU_KGD, SE, SE_1, SE_2, SE_3, SE_4, SI, SK, TR, UA, UA_BEI, UA_DOBTPP, UA_IPS, UK, XK



Enter origin bidding zone (e.g., DK_1, DE):  DK_1



Possible Neighbors for DK_1:
DE_AT_LU, DE_LU, DK_2, GB, NL, NO_2, SE_3



Enter destination bidding zone:  DE_LU



Enter dates using pattern YYYYMMDD (e.g., 20260920)


START date:  20260920
END date:  20260925



⏳ Fetching live data from ENTSO-E...


In [59]:
import plotly.io as pio
# ==========================================================
# STEP 5: Process and Generate Plot (Unified Plotting Engine)
# ==========================================================
df_price = data_results.get('day_ahead_prices')
df_actual = data_results.get('generation')
df_forecast = data_results.get('generation_forecast') # ดึงข้อมูลมาเตรียมพล็อต
df_load = data_results.get('load')
df_net_pos = data_results.get('net_position')
df_flows = data_results.get('crossborder_flows')

if df_price is None or df_actual is None:
    print("\n❌ Error: Essential datasets are missing. Plotting canceled.")
else:
    print("\n📊 Generating Interactive Plotly Dashboard...")
    
    # 1. Process Total Generation Actual & Forecast
    available_cols = [c for c in df_actual.columns if "Consumption" not in c]
    actual_total = df_actual[available_cols].sum(axis=1)
    
    # Process Total Forecast series
    if df_forecast is not None:
        forecast_total = df_forecast.sum(axis=1) if isinstance(df_forecast, pd.DataFrame) else df_forecast
    else:
        forecast_total = None

    # 2. Merit Order logic
    merit_1_all = ['Solar', 'Wind Onshore', 'Wind Offshore', 'Hydro Run-of-river and pondage', 'Marine', 'Nuclear', 'Geothermal']
    merit_2_all = ['Biomass', 'Waste', 'Other renewable', 'Hydro Water Reservoir', 'Hydro Pumped Storage', 'Energy storage']
    merit_3_all = ['Fossil Gas', 'Fossil Hard coal', 'Fossil Brown coal/Lignite', 'Fossil Coal-derived gas', 'Fossil Oil', 'Fossil Oil shale', 'Fossil Peat', 'Other']

    m1_existing = [c for c in merit_1_all if c in available_cols]
    m2_existing = [c for c in merit_2_all if c in available_cols]
    m3_existing = [c for c in merit_3_all if c in available_cols]

    actual_m1 = df_actual[m1_existing].sum(axis=1) if m1_existing else pd.Series(0, index=df_actual.index)
    actual_m2 = df_actual[m2_existing].sum(axis=1) if m2_existing else pd.Series(0, index=df_actual.index)
    actual_m3 = df_actual[m3_existing].sum(axis=1) if m3_existing else pd.Series(0, index=df_actual.index)

    load_series = df_load.iloc[:, 0] if isinstance(df_load, pd.DataFrame) else df_load
    net_pos_series = df_net_pos.iloc[:, 0] if isinstance(df_net_pos, pd.DataFrame) else df_net_pos
    flow_series = df_flows.iloc[:, 0] if isinstance(df_flows, pd.DataFrame) else df_flows

    # 3. Build Plotly Graph
    fig = make_subplots(specs=[[{"secondary_y": True}]])

    # Demand
    if load_series is not None:
        fig.add_trace(go.Scatter(x=load_series.index, y=load_series.values, name="Actual Load (Demand)", line=dict(color="#333333", width=2.5)), secondary_y=False)
    
    # Total Actual Gen
    fig.add_trace(go.Scatter(x=actual_total.index, y=actual_total.values, name="Total Actual Generation", line=dict(color="#000080", width=3)), secondary_y=False)

    # 🔵 Total Generation Forecast Line (New Addition!)
    if forecast_total is not None:
        fig.add_trace(go.Scatter(x=forecast_total.index, y=forecast_total.values, name="Total Generation Forecast", line=dict(color="#7f7fcc", width=2, dash="dash")), secondary_y=False)

    # Net Position & Interconnection Flow
    if net_pos_series is not None:
        fig.add_trace(go.Scatter(x=net_pos_series.index, y=net_pos_series.values, name="Net Position (Export/Import)", line=dict(color="#17becf", width=1.5, dash="dot")), secondary_y=False)
    if flow_series is not None:
        fig.add_trace(go.Scatter(x=flow_series.index, y=flow_series.values, name=f"Flow: {country_code} ➔ {country_code_to}", line=dict(color="#db5f8e", width=2, dash="dashdot")), secondary_y=False)

    # Merit Order Groups Traces
    fig.add_trace(go.Scatter(x=actual_m1.index, y=actual_m1.values, name="1. Variable RES/Nuclear", line=dict(color="#2ca02c", width=1.5)), secondary_y=False)
    fig.add_trace(go.Scatter(x=actual_m2.index, y=actual_m2.values, name="2. Biomass/Waste/Storage", line=dict(color="#9467bd", width=1.5)), secondary_y=False)
    fig.add_trace(go.Scatter(x=actual_m3.index, y=actual_m3.values, name="3. Fossil Fuels/Other", line=dict(color="#d62728", width=1.5)), secondary_y=False)

    # Day-Ahead Price (Right Y-axis)
    fig.add_trace(go.Scatter(x=df_price.index, y=df_price.values, name="Day-Ahead Price", line=dict(color="#ff7f0e", width=2.5)), secondary_y=True)

    fig.update_layout(
        title=dict(text=f"<b>Grid Analysis Dashboard: Zone {country_code} ➔ {country_code_to}</b>", x=0.5, font=dict(size=16)),
        xaxis_title="Timestamp",
        hovermode="x unified",
        legend=dict(x=0.01, y=0.99, bgcolor="rgba(255, 255, 255, 0.7)", font=dict(size=10)),
        width=1300,
        height=850
    )
    fig.update_yaxes(title_text="Electricity Volume (MW)", secondary_y=False, gridcolor="rgba(0,0,0,0.05)")
    fig.update_yaxes(title_text="Day-Ahead Price (EUR/MWh)", secondary_y=True)

    fig.show()
    
    fig.write_html("grid_analysis_dashboard.html")
    print("✅ Saved interactive HTML dashboard!")


📊 Generating Interactive Plotly Dashboard...


✅ Saved interactive HTML dashboard!
